In [0]:
%run ../init_framework

In [0]:
# # Initialize the CDF Read Stream ---
# df_bronze_repayments = (spark.readStream
#     .format("delta")
#     .option("readChangeFeed", "true")
#     .option("startingVersion", 1) 
#     .table(REPAYMENTS_BRONZE))

In [0]:
# Initialize the CDF Read Stream ---
df_bronze_repayments = (spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1) 
    .table(REPAYMENTS_BRONZE))

In [0]:
# --- 1. Define Rename Map ---
repayments_rename_map = {
    "total_rec_prncp": "total_principal_received",
    "total_rec_int": "total_interest_received",
    "total_rec_late_fee": "total_late_fee_received",
    "total_pymnt": "total_payment_received",
    "last_pymnt_amnt": "last_payment_amount",
    "last_pymnt_d": "last_payment_date",
    "next_pymnt_d": "next_payment_date",
}

# --- 2. Execute Transformation ---
df_renamed_repayments = standardize_column_names(
    df_bronze_repayments, repayments_rename_map, strict=True
)

In [0]:
df_with_metadata = apply_silver_metadata(df_renamed_repayments)

In [0]:
df_distinct = df_with_metadata.distinct()

In [0]:
# df_distinct.count()
# df_with_metadata.count()
# # 1507134

In [0]:
# Keep Valid Repayments: Drop NULL or UNKNOWN
col_ls = ["total_principal_received", "total_interest_received", "total_late_fee_received", "total_payment_received", "last_payment_amount"]
df_valid_repayments = drop_critical_nulls(df_distinct, col_ls)

In [0]:
from pyspark.sql.functions import col, when

# Apply repair logic to total_payment_received
df_payment_fixed = df_valid_repayments.withColumn(
    "total_payment_received",
    when(
        (col("total_principal_received") != 0.0)
        & (col("total_payment_received") == 0.0),
        col("total_principal_received")
        + col("total_interest_received")
        + col("total_late_fee_received"),
    ).otherwise(col("total_payment_received")),
)

In [0]:
df_valid_payment = df_payment_fixed.filter("total_payment_received != 0.0")
df_valid_payment.limit(3).display()

In [0]:
from pyspark.sql.functions import col, when, lit

# Replace string "0.0" with a true NULL/None
df_last_payments_fixed = df_valid_payment.withColumn(
  "last_payment_date",
   when(
       col("last_payment_date") == "0.0",
       lit(None).cast("string") 
   ).otherwise(col("last_payment_date"))
)

In [0]:
# df_last_payments_fixed.filter("last_payment_date is NULL").count()

In [0]:
from pyspark.sql.functions import col, when, lit

# Replace string "0.0" with a true NULL/None
df_repayments_final = df_valid_payment.withColumn(
  "next_payment_date",
   when(
       col("next_payment_date") == "0.0",
       lit(None).cast("string") 
   ).otherwise(col("next_payment_date"))
)

In [0]:
df_repayments_final.filter("last_payment_date is NULL").count()